# Build Dashboard
Loads all `.pkl` runs, computes metrics, and generates `dashboard.html`.

In [1]:
import glob
import json
import numpy as np
from plotting import load_run, compute_metrics

# Load all pkl files
pkl_files = sorted(glob.glob("gd_trajectories/run_*.pkl"))
print(f"Found {len(pkl_files)} runs")

# Compute metrics and group by ratio
runs_by_ratio = {}  # ratio -> list of run dicts

for f in pkl_files:
    data = load_run(f)
    metrics = compute_metrics(data)
    if not metrics:
        continue
    
    config = data["config"]
    n, d, k, seed = config["n"], config["d"], config["k"], config["seed"]
    ratio = round(k / n, 6)
    
    run_entry = {
        "label": f"k/n = {ratio:.4f}  (k={k}, n={n}, d={d}, seed={seed})",
        "k": int(k), "n": int(n), "d": int(d), "seed": int(seed),
        "times": metrics["times"].tolist(),
    }
    
    # Add available metrics
    if "norms" in metrics:
        run_entry["norm"] = metrics["norms"].tolist()
    if "loss_values" in metrics:
        run_entry["train_loss"] = metrics["loss_values"].tolist()
        run_entry["loss_times"] = metrics["loss_times"].tolist()
    if "pop_loss_values" in metrics:
        run_entry["pop_loss"] = metrics["pop_loss_values"].tolist()
        run_entry["pop_loss_times"] = metrics["pop_loss_times"].tolist()
    if "angle_w_star" in metrics:
        run_entry["angle_w_star"] = metrics["angle_w_star"].tolist()
    if "angle_w_tilde" in metrics:
        run_entry["angle_w_tilde"] = metrics["angle_w_tilde"].tolist()
    if "stopping_times" in metrics:
        run_entry["stopping_times"] = [int(t) for t in metrics["stopping_times"]]
    if "w_star_norm" in metrics:
        run_entry["w_star_norm"] = float(metrics["w_star_norm"])
    
    if ratio not in runs_by_ratio:
        runs_by_ratio[ratio] = []
    runs_by_ratio[ratio].append(run_entry)

sorted_ratios = sorted(runs_by_ratio.keys())
print(f"Ratios: {sorted_ratios}")
print(f"Seeds per ratio: { {r: [run['seed'] for run in runs_by_ratio[r]] for r in sorted_ratios} }")

Found 14 runs
Ratios: [0.05, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5]
Seeds per ratio: {0.05: [0, 1, 2, 4], 0.1: [0, 2], 0.25: [2], 0.5: [2, 4], 0.75: [2], 1.0: [2], 1.5: [2], 2.0: [2], 2.5: [2]}


In [2]:
# Build the JSON data blob — store all runs as a flat list
all_runs = []
for ratio in sorted(runs_by_ratio.keys()):
    all_runs.extend(runs_by_ratio[ratio])

dashboard_data = {"runs": all_runs}
json_blob = json.dumps(dashboard_data)
print(f"JSON size: {len(json_blob) / 1024:.1f} KB")
print(f"Total runs: {len(all_runs)}")

JSON size: 334.5 KB
Total runs: 14


In [3]:
html_template = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>GD Trajectory Dashboard</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; background: #f5f5f5; padding: 20px; }
  h1 { text-align: center; margin-bottom: 18px; font-size: 1.5em; color: #333; }
  .controls { background: #fff; border-radius: 8px; padding: 18px 24px; margin-bottom: 16px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); }
  .slider-row { display: flex; align-items: center; gap: 8px; margin-bottom: 12px; }
  .slider-row label { font-weight: 600; white-space: nowrap; }
  .slider-row input[type=range] { flex: 1; }
  .slider-row .slider-label { min-width: 340px; font-size: 0.95em; color: #555; }
  .step-btn { width: 32px; height: 32px; font-size: 1.1em; font-weight: 700; cursor: pointer; border: 1px solid #ccc; border-radius: 4px; background: #fff; color: #333; display: flex; align-items: center; justify-content: center; }
  .step-btn:hover { background: #eee; }
  .step-btn:active { background: #ddd; }
  .controls-row { display: flex; align-items: center; gap: 24px; flex-wrap: wrap; margin-bottom: 8px; }
  .controls-row label { font-weight: 600; margin-right: 4px; }
  .checkbox-group { display: flex; gap: 14px; flex-wrap: wrap; }
  .checkbox-group label { font-weight: 400; cursor: pointer; display: flex; align-items: center; gap: 4px; }
  .ctrl-select { padding: 4px 8px; font-size: 0.95em; }
  .plots { display: flex; gap: 16px; }
  .plots > div { flex: 1; background: #fff; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); padding: 8px; }
  @media (max-width: 900px) { .plots { flex-direction: column; } }
</style>
</head>
<body>
<h1>GD Trajectory Dashboard</h1>

<div class="controls">
  <div class="controls-row">
    <div>
      <label>n:</label>
      <select id="nSelect" class="ctrl-select"></select>
    </div>
    <div>
      <label>d:</label>
      <select id="dSelect" class="ctrl-select"></select>
    </div>
    <div>
      <label>Seed:</label>
      <select id="seedSelect" class="ctrl-select"></select>
    </div>
  </div>
  <div class="slider-row">
    <label>k/n ratio:</label>
    <button class="step-btn" id="btnPrev">&lt;</button>
    <input type="range" id="ratioSlider" min="0" max="0" value="0" step="1">
    <button class="step-btn" id="btnNext">&gt;</button>
    <span class="slider-label" id="sliderLabel">—</span>
  </div>
  <div class="controls-row">
    <div class="checkbox-group" id="metricToggles"></div>
    <div><label>X-axis:</label> <button class="step-btn" id="btnScale" style="width:auto;padding:0 10px;font-size:0.9em;font-weight:600;">Log</button></div>
    <div style="display:flex;align-items:center;gap:6px;"><label>t range:</label> <input type="text" id="tMin" class="ctrl-select" style="width:80px;" placeholder="min" value="0"> — <input type="text" id="tMax" class="ctrl-select" style="width:80px;" placeholder="max"> <button class="step-btn" id="btnApplyRange" style="width:auto;padding:0 8px;font-size:0.8em;">Apply</button> <button class="step-btn" id="btnResetRange" style="width:auto;padding:0 8px;font-size:0.8em;">Reset</button></div>
  </div>
</div>

<div class="plots">
  <div><div id="plotLeft" style="width:100%;height:500px;"></div></div>
  <div><div id="plotRight" style="width:100%;height:500px;"></div></div>
</div>

<script>
const DATA = __JSON_DATA__;

const METRICS = [
  { key: 'norm',        label: 'Norm',         color: '#2ca02c', timesKey: 'times' },
  { key: 'train_loss',  label: 'Train Loss',   color: '#d62728', timesKey: 'loss_times' },
  { key: 'pop_loss',    label: 'Test Loss',     color: '#9467bd', timesKey: 'pop_loss_times' },
  { key: 'angle_w_star',label: 'Angle to w*',  color: '#1f77b4', timesKey: 'times', unnormalized: true },
  { key: 'angle_w_tilde',label: 'Angle to w\u0303', color: '#ff7f0e', timesKey: 'times', unnormalized: true },
];

const slider = document.getElementById('ratioSlider');
const sliderLabel = document.getElementById('sliderLabel');
const nSelect = document.getElementById('nSelect');
const dSelect = document.getElementById('dSelect');
const seedSelect = document.getElementById('seedSelect');
const togglesDiv = document.getElementById('metricToggles');
const btnPrev = document.getElementById('btnPrev');
const btnNext = document.getElementById('btnNext');
const btnScale = document.getElementById('btnScale');
let xLog = true;
const tMin = document.getElementById('tMin');
const tMax = document.getElementById('tMax');
const btnApplyRange = document.getElementById('btnApplyRange');
const btnResetRange = document.getElementById('btnResetRange');

let prevRatioIdx = null;
let currentRatioIdx = 0;
let filteredRatios = [];
let filteredRuns = {};

// Build checkboxes
METRICS.forEach(m => {
  const lbl = document.createElement('label');
  const cb = document.createElement('input');
  cb.type = 'checkbox'; cb.checked = true; cb.dataset.metric = m.key;
  cb.addEventListener('change', updatePlots);
  lbl.appendChild(cb);
  lbl.appendChild(document.createTextNode(' ' + m.label));
  togglesDiv.appendChild(lbl);
});

// Extract unique n and d values
const allN = [...new Set(DATA.runs.map(r => r.n))].sort((a, b) => a - b);
const allD = [...new Set(DATA.runs.map(r => r.d))].sort((a, b) => a - b);

allN.forEach(n => {
  const opt = document.createElement('option');
  opt.value = n; opt.textContent = n;
  nSelect.appendChild(opt);
});
allD.forEach(d => {
  const opt = document.createElement('option');
  opt.value = d; opt.textContent = d;
  dSelect.appendChild(opt);
});

const ndCounts = {};
DATA.runs.forEach(r => {
  const key = r.n + ',' + r.d;
  ndCounts[key] = (ndCounts[key] || 0) + 1;
});
const bestND = Object.entries(ndCounts).sort((a, b) => b[1] - a[1])[0][0].split(',');
nSelect.value = bestND[0];
dSelect.value = bestND[1];

nSelect.addEventListener('change', onNDChange);
dSelect.addEventListener('change', onNDChange);
seedSelect.addEventListener('change', updatePlots);
slider.addEventListener('input', onSliderChange);

btnPrev.addEventListener('click', function() {
  const newVal = Math.max(0, parseInt(slider.value) - 1);
  slider.value = newVal;
  onSliderChange();
});
btnNext.addEventListener('click', function() {
  const newVal = Math.min(parseInt(slider.max), parseInt(slider.value) + 1);
  slider.value = newVal;
  onSliderChange();
});

btnApplyRange.addEventListener('click', updatePlots);
tMin.addEventListener('keydown', function(e) { if (e.key === 'Enter') updatePlots(); });
tMax.addEventListener('keydown', function(e) { if (e.key === 'Enter') updatePlots(); });
btnResetRange.addEventListener('click', function() {
  tMin.value = '0';
  tMax.value = '';
  updatePlots();
});

btnScale.addEventListener('click', function() {
  xLog = !xLog;
  btnScale.textContent = xLog ? 'Log' : 'Linear';
  updatePlots();
});

function getCheckedMetrics() {
  return Array.from(togglesDiv.querySelectorAll('input:checked')).map(cb => cb.dataset.metric);
}

function onNDChange() {
  const n = parseInt(nSelect.value);
  const d = parseInt(dSelect.value);

  filteredRuns = {};
  DATA.runs.forEach(r => {
    if (r.n !== n || r.d !== d) return;
    const ratio = round(r.k / r.n);
    if (!filteredRuns[ratio]) filteredRuns[ratio] = [];
    filteredRuns[ratio].push(r);
  });
  filteredRatios = Object.keys(filteredRuns).map(Number).sort((a, b) => a - b);

  prevRatioIdx = null;
  currentRatioIdx = 0;
  slider.min = 0;
  slider.max = Math.max(0, filteredRatios.length - 1);
  slider.value = 0;

  populateSeeds(0);
  updateSliderLabel();
  updatePlots();
}

function round(v) {
  return Math.round(v * 1e6) / 1e6;
}

function populateSeeds(ratioIdx) {
  const runs = getRunsForRatio(ratioIdx);
  const prevSeed = seedSelect.value;
  seedSelect.innerHTML = '';
  runs.forEach((run, i) => {
    const opt = document.createElement('option');
    opt.value = i;
    opt.textContent = 'seed=' + run.seed;
    seedSelect.appendChild(opt);
  });
  const matchIdx = runs.findIndex(r => String(r.seed) === prevSeed);
  seedSelect.value = matchIdx >= 0 ? matchIdx : 0;
}

function getRunsForRatio(ratioIdx) {
  if (ratioIdx < 0 || ratioIdx >= filteredRatios.length) return [];
  const ratio = filteredRatios[ratioIdx];
  return filteredRuns[ratio] || [];
}

function getRun(ratioIdx) {
  const runs = getRunsForRatio(ratioIdx);
  if (runs.length === 0) return null;
  if (ratioIdx !== currentRatioIdx) {
    const currentRuns = getRunsForRatio(currentRatioIdx);
    const selIdx = parseInt(seedSelect.value) || 0;
    if (currentRuns[selIdx]) {
      const targetSeed = currentRuns[selIdx].seed;
      const match = runs.find(r => r.seed === targetSeed);
      if (match) return match;
    }
  }
  const idx = Math.min(parseInt(seedSelect.value) || 0, runs.length - 1);
  return runs[idx];
}

function normalize(arr) {
  if (!arr || arr.length === 0) return { normed: [], mn: 0, mx: 0 };
  const mn = Math.min(...arr);
  const mx = Math.max(...arr);
  if (mx === mn) return { normed: arr.map(() => 0.5), mn, mx };
  return { normed: arr.map(v => (v - mn) / (mx - mn)), mn, mx };
}

function makeTitle(run, prefix) {
  if (!run) return prefix + ': (no data)';
  const ratio = round(run.k / run.n);
  return prefix + ': k/n=' + ratio.toFixed(4) + '  (k=' + run.k + ', n=' + run.n + ', d=' + run.d + ', seed=' + run.seed + ')';
}

function buildTraces(run, checkedMetrics) {
  if (!run) return [];
  const traces = [];
  METRICS.forEach(m => {
    if (!checkedMetrics.includes(m.key)) return;
    if (!run[m.key]) return;
    const raw = run[m.key];
    const times = run[m.timesKey] || run.times;
    const useRaw = !!m.unnormalized;
    const { normed, mn, mx } = useRaw ? { normed: raw, mn: Math.min(...raw), mx: Math.max(...raw) } : normalize(raw);
    traces.push({
      x: times,
      y: normed,
      type: 'scatter',
      mode: 'lines',
      name: m.label,
      line: { color: m.color, width: 2 },
      text: raw.map(v => m.label + ' = ' + v.toFixed(4)),
      hovertemplate: '%{text}<br>t = %{x}<extra></extra>',
    });

    if (m.key === 'norm' && run.w_star_norm != null && mx !== mn) {
      const wStarNormed = (run.w_star_norm - mn) / (mx - mn);
      traces.push({
        x: [times[0], times[times.length - 1]],
        y: [wStarNormed, wStarNormed],
        type: 'scatter',
        mode: 'lines',
        name: '||w*|| = ' + run.w_star_norm.toFixed(2),
        line: { color: 'gray', width: 1.5, dash: 'dash' },
        hovertemplate: '||w*|| = ' + run.w_star_norm.toFixed(4) + '<extra></extra>',
      });
    }
  });
  if (run.stopping_times) {
    run.stopping_times.forEach((st, i) => {
      traces.push({
        x: [st, st], y: [-10, 200],
        type: 'scatter', mode: 'lines',
        line: { color: 'red', width: 1.5, dash: 'dash' },
        showlegend: i === 0,
        name: i === 0 ? 't* = ' + st : '',
        hoverinfo: 'skip',
      });
    });
  }
  return traces;
}

function parseValue(str) {
  if (!str || str.trim() === '') return null;
  str = str.trim().toLowerCase();
  const multipliers = {'k': 1e3, 'm': 1e6};
  const last = str[str.length - 1];
  if (multipliers[last]) return parseFloat(str.slice(0, -1)) * multipliers[last];
  return parseFloat(str);
}

function getXRange() {
  const lo = parseValue(tMin.value);
  const hi = parseValue(tMax.value);
  if (lo === null && hi === null) return undefined;
  if (xLog) {
    const logLo = lo !== null ? Math.log10(Math.max(lo, 1)) : null;
    const logHi = hi !== null ? Math.log10(hi) : null;
    if (logLo !== null && logHi !== null) return [logLo, logHi];
    if (logLo !== null) return [logLo, undefined];
    return [undefined, logHi];
  }
  return [lo, hi];
}

function plotLayout(title) {
  return {
    title: { text: title, font: { size: 13 } },
    xaxis: { title: 'Iteration t', type: xLog ? 'log' : 'linear', gridcolor: '#eee', range: getXRange() },
    yaxis: { title: 'Value', gridcolor: '#eee' },
    legend: { orientation: 'h', y: -0.15, x: 0.5, xanchor: 'center' },
    margin: { t: 50, b: 80, l: 50, r: 20 },
    plot_bgcolor: '#fafafa',
    hovermode: 'closest',
  };
}

function updateSliderLabel() {
  const run = getRun(currentRatioIdx);
  if (run) {
    const ratio = round(run.k / run.n);
    sliderLabel.textContent = 'k/n = ' + ratio.toFixed(4) + '  (k=' + run.k + ')';
  } else {
    sliderLabel.textContent = '—';
  }
}

function updatePlots() {
  const checked = getCheckedMetrics();

  const currentRun = getRun(currentRatioIdx);
  Plotly.react('plotLeft', buildTraces(currentRun, checked),
    plotLayout(makeTitle(currentRun, 'Current')), {responsive: true});

  if (prevRatioIdx !== null && prevRatioIdx !== currentRatioIdx) {
    const prevRun = getRun(prevRatioIdx);
    Plotly.react('plotRight', buildTraces(prevRun, checked),
      plotLayout(makeTitle(prevRun, 'Previous')), {responsive: true});
  } else {
    Plotly.react('plotRight', [], plotLayout('Previous: (none)'), {responsive: true});
  }
}

function onSliderChange() {
  const newIdx = parseInt(slider.value);
  if (newIdx !== currentRatioIdx) {
    prevRatioIdx = currentRatioIdx;
    currentRatioIdx = newIdx;
  }
  populateSeeds(currentRatioIdx);
  updateSliderLabel();
  updatePlots();
}

// Initial render
onNDChange();
</script>
</body>
</html>"""

# Inject JSON data
html_content = html_template.replace('__JSON_DATA__', json_blob)

# Write to file
output_path = "dashboard.html"
with open(output_path, "w") as f:
    f.write(html_content)

print(f"Written {output_path} ({len(html_content) / 1024:.1f} KB")

Written dashboard.html (347.6 KB
